## E1 — Test-set reconstruction + KL usage (Kaggle)\n\nEvaluates trained checkpoints on the **test memmap split** for all three model families:\n- Plain Transformer (reconstruction only)\n- Simple VAE (reconstruction + KL)\n- Hierarchical VAE (reconstruction + KL)\n\nOutputs are written under `artifacts/eval_recon_kl/`:\n- `recon_kl_summary.json`\n- `recon_kl_table.md`\n- `kl_bits_hist_*.png`\n- `recon_ppl_bar.png`\n\n### Protocol (simple + reproducible)\n- Teacher-forced evaluation on the held-out test split (PAD ignored).\n- VAEs report **true KL bits/dim** + **active units** (not clamped by free-bits).\n

In [ ]:
from __future__ import annotations

import json
import math
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('torch cuda:', torch.version.cuda)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

IS_KAGGLE = Path('/kaggle').exists() and Path('/kaggle/input').exists()
INPUT_ROOT = Path('/kaggle/input') if IS_KAGGLE else Path.cwd().parents[0]
WORKING_ROOT = Path('/kaggle/working') if IS_KAGGLE else (Path.cwd() / 'artifacts' / 'local_eval')
OUT_DIR = (WORKING_ROOT / 'artifacts' / 'eval_recon_kl').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('IS_KAGGLE:', IS_KAGGLE)
print('INPUT_ROOT:', INPUT_ROOT)
print('WORKING_ROOT:', WORKING_ROOT)
print('OUT_DIR:', OUT_DIR)

# Repo layout: ml/src/musicgen
import sys
repo_root = (Path.cwd().parents[1] if (Path.cwd().name == 'notebooks') else Path.cwd())
ml_src = (repo_root / 'ml' / 'src').resolve()
if str(ml_src) not in sys.path:
    sys.path.insert(0, str(ml_src))

from musicgen.inference.tokenizer import load_remi_tokenizer
from musicgen.inference.service import _load_checkpoint as load_ckpt  # type: ignore
from musicgen.inference.service import _build_model as build_model  # type: ignore
from musicgen.inference.service import _resolve_ckpt_path as resolve_ckpt  # type: ignore


In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    name: str
    model_type: str  # 'plain' | 'vae' | 'simple_vae'
    ckpt_path: str
    block_size: int = 1024
    bars_per_sample: int = 8


# Update these as needed.
SPECS = [
    ModelSpec('plain_v3', 'plain', 'ml/src/musicgen/runs/plain_v3/ckpt.pt', 1024),
    ModelSpec('simple_v2', 'simple_vae', 'ml/src/musicgen/runs/simple_v2/ckpt.pt', 1024),
    ModelSpec('vae_v3', 'vae', 'ml/src/musicgen/runs/vae_v3/ckpt.pt', 1024),
]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TOK = load_remi_tokenizer()
PAD_ID = int(TOK.pad_id)
VOCAB = 195

print('device:', DEVICE)
print('pad_id:', PAD_ID)


In [ ]:
# Minimal memmap dataset loader (matches the spec: X uint16 [N,1024]).
# Expects artifacts/preprocessed_memmap/test/{X.dat,attributes.dat,bar_indices.dat,meta.json}

class MemmapTestSet(Dataset):
    def __init__(self, root: Path):
        self.root = root
        meta = json.loads((root / 'meta.json').read_text(encoding='utf-8'))
        self.n = int(meta['shapes']['X'][0])
        self.block = int(meta['shapes']['X'][1])
        self.X = np.memmap(root / 'X.dat', dtype=np.uint16, mode='r', shape=tuple(meta['shapes']['X']))

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, idx: int):
        x = np.asarray(self.X[idx], dtype=np.int64)
        return torch.from_numpy(x)


TEST_ROOT = (repo_root / 'artifacts' / 'preprocessed_memmap' / 'test').resolve()
assert TEST_ROOT.exists(), f'Missing test memmap split at {TEST_ROOT}'
ds = MemmapTestSet(TEST_ROOT)
print('test size:', len(ds), 'block:', ds.block)

dl = DataLoader(ds, batch_size=16, shuffle=True, num_workers=0, drop_last=True)


In [ ]:
def shift_targets(x: torch.Tensor, pad_id: int) -> torch.Tensor:
    # y = x shifted left by 1; last token becomes PAD
    y = torch.empty_like(x)
    y[:, :-1] = x[:, 1:]
    y[:, -1] = int(pad_id)
    return y


def kl_bits_per_dim(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    # KL(q||p) for diagonal Gaussian; return bits per dimension
    kl_nats = 0.5 * (mu.pow(2) + logvar.exp() - 1.0 - logvar)
    kl_bits = kl_nats / math.log(2.0)
    return kl_bits.mean(dim=0)  # [z_dim]


@torch.no_grad()
def eval_one(spec: ModelSpec, max_batches: int = 200) -> dict[str, Any]:
    ckpt_path = resolve_ckpt(spec.ckpt_path)
    ckpt = load_ckpt(ckpt_path)
    model = build_model(spec.model_type, ckpt, block_size=spec.block_size, bars=spec.bars_per_sample)
    model.to(DEVICE)
    model.eval()

    recon_sum = 0.0
    tok_count = 0
    kl_dim_accum: Optional[torch.Tensor] = None
    nb = 0

    for x in tqdm(dl, total=max_batches, desc=spec.name):
        nb += 1
        x = x.to(DEVICE)
        if x.shape[1] != spec.block_size:
            x = x[:, : spec.block_size]
        y = shift_targets(x, PAD_ID)

        # Dummy conditioning (attributes + bar_indices) for compatibility.
        bar_indices = torch.zeros_like(x, dtype=torch.long)
        attrs = torch.full((x.shape[0], spec.bars_per_sample, 4), 3, dtype=torch.long, device=DEVICE)

        if spec.model_type == 'plain':
            out = model(x, bar_indices=bar_indices, attributes=attrs)
        elif spec.model_type == 'simple_vae':
            # simple vae forward accepts z_p; use internal sampling by passing None
            out = model(x, bar_indices=bar_indices, attributes=attrs, z_p=None)
        else:  # 'vae'
            out = model(x, bar_indices=bar_indices, attributes=attrs, z_p=None)

        logits = out['logits'] if isinstance(out, dict) else out.logits
        loss = F.cross_entropy(
            logits.reshape(-1, VOCAB),
            y.reshape(-1),
            ignore_index=PAD_ID,
            reduction='sum',
        )
        recon_sum += float(loss.item())
        tok_count += int((y != PAD_ID).sum().item())

        if spec.model_type in ('vae', 'simple_vae'):
            # Expect encoder outputs in out if training model returns them.
            mu = out.get('mu') if isinstance(out, dict) else None
            logvar = out.get('logvar') if isinstance(out, dict) else None
            if mu is not None and logvar is not None:
                kld = kl_bits_per_dim(mu, logvar).detach().cpu()
                kl_dim_accum = kld if kl_dim_accum is None else (kl_dim_accum + kld)

        if nb >= max_batches:
            break

    ce = recon_sum / max(1, tok_count)
    ppl = float(math.exp(ce))
    res: dict[str, Any] = {'name': spec.name, 'model_type': spec.model_type, 'ckpt': spec.ckpt_path, 'ce': ce, 'ppl': ppl, 'tokens': tok_count}
    if kl_dim_accum is not None:
        kl_dim = (kl_dim_accum / nb).numpy().tolist()
        res['kl_bits_dim'] = kl_dim
        res['kl_bits_dim_mean'] = float(np.mean(kl_dim))
    return res


results = [eval_one(s, max_batches=200) for s in SPECS]
out_path = OUT_DIR / 'recon_kl_summary.json'
out_path.write_text(json.dumps(results, indent=2), encoding='utf-8')
out_path
